参数管理

In [1]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))

X = torch.rand((2, 4))
net(X)

tensor([[-0.4263],
        [-0.4090]], grad_fn=<AddmmBackward0>)

参数访问

In [2]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[ 0.1288,  0.3087,  0.0108, -0.2335,  0.2487, -0.0898, -0.1268, -0.3125]])), ('bias', tensor([-0.3290]))])


目标参数

In [3]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([-0.3290], requires_grad=True)
tensor([-0.3290])


In [4]:
net[2].weight.grad == None

True

一次性访问所有参数

In [5]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [6]:
net.state_dict(), net.state_dict()['2.bias'].data

(OrderedDict([('0.weight',
               tensor([[-0.4554, -0.4463,  0.4654,  0.1136],
                       [ 0.3965, -0.3374,  0.1822, -0.0976],
                       [ 0.1038, -0.2646,  0.1813, -0.2332],
                       [-0.3745,  0.2993, -0.0178, -0.4729],
                       [-0.0561,  0.4210, -0.1186, -0.3679],
                       [ 0.2743,  0.1951,  0.0751,  0.0386],
                       [ 0.0799,  0.3730, -0.1089, -0.4671],
                       [ 0.2072, -0.4882, -0.0281, -0.0246]])),
              ('0.bias',
               tensor([-0.1859, -0.3979,  0.0972,  0.3683,  0.1009, -0.1990,  0.4956, -0.2338])),
              ('2.weight',
               tensor([[ 0.1288,  0.3087,  0.0108, -0.2335,  0.2487, -0.0898, -0.1268, -0.3125]])),
              ('2.bias', tensor([-0.3290]))]),
 tensor([-0.3290]))

从嵌套块收集参数

In [7]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block {i}', block1())
    return net
rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[-0.3040],
        [-0.3040]], grad_fn=<AddmmBackward0>)

In [8]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


内置初始化

In [9]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean = 0, std = 0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([ 0.0155,  0.0143, -0.0269,  0.0251]), tensor(0.))

In [10]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

对某些块应用不同的初始化方法

In [ ]:
def xavier(m): # 防止前向传递和反向传播的梯度不会出现爆炸或消失
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(xavier)
net[2].apply(init_42)
net[0].weight.data[0], net[2].weight.data

(tensor([ 0.6838,  0.1281,  0.1840, -0.3459]),
 tensor([[42., 42., 42., 42., 42., 42., 42., 42.]]))

自定义初始化

In [12]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape) for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5
net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[9.1350, -0.0000, 0.0000, 6.6989],
        [-0.0000, -0.0000, 0.0000, 0.0000]], grad_fn=<SliceBackward0>)

In [13]:
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.0000,  1.0000,  1.0000,  7.6989])

参数绑定

In [14]:
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), shared, nn.ReLU(), shared, nn.ReLU(), nn.Linear(8, 1))
net(X)
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])
